# Ataxx Zero - Kaggle v11

**Antes de Run All:**
1. Settings -> Accelerator -> **GPU T4 x2**.
2. Add-ons -> Secrets -> agregar `HF_TOKEN` y, opcionalmente, `WANDB_API_KEY`.
3. Revisar solo la celda **Run config**.

El notebook se mantiene chico: clona `main`, arma `v11_pretrain.npz` desde replays versionadas (tournament AIEP + play_sessions casuales), carga `kaggle/v11_config.json` y ejecuta `train.py` como subprocess. v11 corre desde scratch (sin bootstrap) con arch nueva (d_model=192, 8 layers), pretrain humano, replay humano permanente al 20% por batch, symmetry aug D4 y count head auxiliar.


In [ ]:
# === Run config (lo unico que deberia cambiar entre runs) ===

# RUN_NAME / hf_run_id viven en kaggle/v11_config.json — single source of truth.
# Si querés bumpear de v11_2 a v11_3, edita el json en el repo y commitealo.
BASE_CONFIG_PATH = "kaggle/v12_config.json"

# Data curada versionada en el repo. v11 usa el mismo NPZ para pretrain y
# para el replay humano permanente (mismas partidas, distinto uso).
CURATED_SOURCE = "tournament_replays"
CURATED_OUTPUT = "/kaggle/working/data/curated/v11_pretrain.npz"
HUMAN_REPLAY_OUTPUT = "/kaggle/working/data/curated/v11_human_replay.npz"
HUMAN_OVERSAMPLE = 4

# Hardware Kaggle
TRAINER_DEVICES = 2
TRAINER_STRATEGY = "ddp_spawn"
TRAINER_PRECISION = "16-mixed"
SELFPLAY_WORKERS = 2

# Persistencia
HF_REPO_ID = "dieg0code/ataxx-zero"
KEEP_LAST_N_LOCAL = 3


In [ ]:
# === Clonar / actualizar el repo ===

REPO_URL = "https://github.com/Dieg0Code/ataxx-zero-ai.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/ataxx-zero-ai"

import os, subprocess
from pathlib import Path

if not Path(WORKDIR).exists():
    subprocess.check_call(["git", "clone", REPO_URL, WORKDIR])
else:
    subprocess.check_call(["git", "-C", WORKDIR, "fetch", "--prune"])
subprocess.check_call(["git", "-C", WORKDIR, "checkout", BRANCH])
subprocess.check_call(["git", "-C", WORKDIR, "pull", "--ff-only"])

os.chdir(WORKDIR)
subprocess.check_call(["git", "-C", WORKDIR, "log", "-1", "--oneline"])

In [ ]:
# === Instalar dependencias (uv + grupo train) ===

!python -m pip install -q uv
!uv sync --frozen --group train
!uv run python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'devices', torch.cuda.device_count())"

In [ ]:
# === HF Token desde Kaggle Secrets ===

HF_ENABLED = False
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN").strip()
    if not token:
        raise RuntimeError("HF_TOKEN secret está vacío.")
    os.environ["HF_TOKEN"] = token
    HF_ENABLED = True
    print(f"HF_TOKEN cargado ({len(token)} chars). HF persistence ENABLED.")
except Exception as exc:
    print(f"Sin HF_TOKEN en Kaggle Secrets: {exc}")
    print("HF persistence DISABLED — los checkpoints solo van a quedar locales y se pierden al cerrar la sesión.")
# W&B es opcional: si el secret existe, Lightning loggea al proyecto configurado.
try:
    wandb_token = UserSecretsClient().get_secret("WANDB_API_KEY").strip()
    if wandb_token:
        os.environ["WANDB_API_KEY"] = wandb_token
        print(f"WANDB_API_KEY cargado ({len(wandb_token)} chars). W&B logging ENABLED.")
except Exception as exc:
    print(f"Sin WANDB_API_KEY en Kaggle Secrets: {exc}")


In [ ]:
# === Dataset curado para pretraining + human replay ===

from pathlib import Path
import shutil
import subprocess

Path(CURATED_OUTPUT).parent.mkdir(parents=True, exist_ok=True)
subprocess.check_call([
    "uv", "run", "python", "scripts/curate_training_data.py",
    "--source", CURATED_SOURCE,
    "--output", CURATED_OUTPUT,
    "--human-oversample", str(HUMAN_OVERSAMPLE),
])

# v11 reusa el mismo NPZ para el human replay buffer permanente.
shutil.copy(CURATED_OUTPUT, HUMAN_REPLAY_OUTPUT)
print(f"Human replay NPZ: {HUMAN_REPLAY_OUTPUT}")


In [ ]:
# === Detectar GPUs y escribir run_config.json ===

import json
import torch

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    names = [torch.cuda.get_device_name(i) for i in range(n_gpus)]
    print(f"GPUs detectadas: {n_gpus} ({', '.join(names)})")
    effective_devices = min(TRAINER_DEVICES, n_gpus)
    effective_strategy = TRAINER_STRATEGY if effective_devices > 1 else "auto"
else:
    print("Sin CUDA: revisa Accelerator en Settings antes de entrenar.")
    effective_devices = 1
    effective_strategy = "auto"

with open(BASE_CONFIG_PATH, "r", encoding="utf-8") as fh:
    run_config = json.load(fh)

# RUN_NAME viene del json — single source of truth. NO sobrescribir hf_run_id
# desde el notebook: si lo hacés acá, cualquier bump en el json se pierde y
# todos los runs terminan apilados en la misma carpeta de HF (lección PM09).
RUN_NAME = run_config["hf_run_id"]

run_config.update({
    "hf_enabled": HF_ENABLED,
    "hf_repo_id": HF_REPO_ID,
    "trainer_devices": effective_devices,
    "trainer_strategy": effective_strategy,
    "trainer_precision": TRAINER_PRECISION,
    "selfplay_workers": SELFPLAY_WORKERS,
    "pretrain_dataset_path": CURATED_OUTPUT,
    "human_replay_path": HUMAN_REPLAY_OUTPUT,
    "hf_local_dir": "/kaggle/working/hf_checkpoints",
    "keep_last_n_local_checkpoints": KEEP_LAST_N_LOCAL,
    "keep_last_n_hf_checkpoints": KEEP_LAST_N_LOCAL,
})

config_path = "/kaggle/working/run_config.json"
with open(config_path, "w", encoding="utf-8") as fh:
    json.dump(run_config, fh, indent=2)

print(f"Run          : {RUN_NAME}")
print(f"Config       : {BASE_CONFIG_PATH} -> {config_path}")
print(f"Pretrain data: {CURATED_OUTPUT}")
print(f"Human replay : {HUMAN_REPLAY_OUTPUT} (fraction={run_config['human_batch_fraction']}, value_mask={run_config['human_value_mask']})")
print(f"Scope        : {run_config['iterations']} iters, {run_config['episodes_per_iter']} eps/iter, {run_config['mcts_sims']} sims")
print(f"Arch         : d_model={run_config['d_model']} layers={run_config['num_layers']} value_depth={run_config['value_head_depth']} count_head={run_config['count_head_enabled']} sym_aug={run_config['symmetry_augmentation']}")
print(f"Gate         : baseline={run_config['baseline_checkpoint']} composite={run_config['baseline_composite']} h2h>={run_config['baseline_h2h_min_score']} min_iter={run_config['eval_absolute_min_iteration']}")
print(f"Hardware     : devices={effective_devices}, strategy={effective_strategy}, precision={TRAINER_PRECISION}")
print(f"HF           : {HF_REPO_ID if HF_ENABLED else 'disabled'}")


In [ ]:
# === Entrenar ===
# train.py corre como subprocess (no `import train`) para que ddp_spawn pueda
# re-ejecutar el script en cada worker. Llamar train.main() directo desde el
# notebook hace que los workers spawned pidan re-ejecutar el kernel launcher
# de Jupyter, que rechaza los args de Lightning con SystemExit: 2.

extra = ["--hf"] if HF_ENABLED else []
subprocess.check_call([
    "uv", "run", "python", "train.py",
    "--config-json", config_path,
    *extra,
])

In [ ]:
# === Post-run: historial a CSV ===

if HF_ENABLED:
    !uv run python scripts/fetch_run_history.py {RUN_NAME}
    print(f"CSV en runs_history/{RUN_NAME}/{RUN_NAME}_history.csv")
else:
    print("HF desactivado: no hay metadata remota para descargar.")
